# PSM 50M Product-Safe Gate

Use this notebook to verify the current best 50M checkpoint in Colab. It does not continue the failed auxiliary action-head branch.

## 1. Setup

Run in a GPU runtime. If the repo is already mounted or cloned, skip the clone line.

In [ ]:
!pip install -U huggingface_hub safetensors
![ -d /content/PSM ] || git clone https://github.com/chkrishna/PSM.git /content/PSM
%cd /content/PSM

## 2. Verify CUDA

In [ ]:
import json, torch
print(json.dumps({
    "torch": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_version": torch.version.cuda,
    "device_count": torch.cuda.device_count(),
    "name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}, indent=2))
assert torch.cuda.is_available(), "Switch Colab runtime to GPU before evaluation or training."

## 3. Optional Artifact Download

Set `HF_REPO` to the artifact repo that contains `psm-model/checkpoints/real-v2-50m-concept-repair-step-005300.pt`, tokenizer artifacts, and probe datasets. Leave it empty if files are already present.

In [ ]:
HF_REPO = ""
if HF_REPO:
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id=HF_REPO, local_dir="/content/PSM", local_dir_use_symlinks=False)


## 4. Required Files

In [ ]:
from pathlib import Path
required = [
    "psm-model/checkpoints/real-v2-50m-concept-repair-step-005300.pt",
    "psm-model/checkpoints/real-v2-50m-concept-repair-step-005300.tokenizer.json",
    "psm-model/checkpoints/real-v2-50m-concept-repair-step-005300.meta.json",
    "psm-model/data/action-foundation-v1/action-probe.jsonl",
    "psm-model/data/concept-curriculum-v1/action-probe.jsonl",
    "psm-model/data/fast-mixed-10k-ctx2048/action-probe-20.jsonl",
    "psm-model/data/direct-behavior-v1/manual-probe.jsonl",
]
missing = [path for path in required if not Path(path).exists()]
print(json.dumps({"missing": missing}, indent=2))
assert not missing, "Download or mount the missing artifacts before continuing."

## 5. Diagnostic Gate

This is expected to fail today because it includes model-only action selection thresholds.

In [ ]:
CHECKPOINT = "psm-model/checkpoints/real-v2-50m-concept-repair-step-005300.pt"
!PYTHONPATH=psm-model/src python -m psm_model.eval_checkpoint_summary {CHECKPOINT} --device auto

## 6. Product-Safe Gate

This checks the constrained `safe_generate` path: calibrated action plus schema-valid extractive fields.

In [ ]:
!PYTHONPATH=psm-model/src python -m psm_model.gate_checkpoint {CHECKPOINT} --device auto --mode product-safe

## 7. Direct Text Smoke

In [ ]:
import json, os, subprocess, sys
payloads = [
    {"conversation":"User: I prefer concise technical answers.","operation":"remember"},
    {"conversation":"User: Today I met Dana at 3pm to review the PSM roadmap.","operation":"remember","source_timestamp":"2026-06-03T00:00:00Z"},
    {"conversation":"User: okay thanks haha and the weather outside is cloudy.","operation":"remember"},
]
for payload in payloads:
    raw = json.dumps(payload)
    print("INPUT", raw)
    env = dict(os.environ, PYTHONPATH="psm-model/src")
    result = subprocess.run([sys.executable, "-m", "psm_model.safe_generate", CHECKPOINT, raw, "--device", "auto"], env=env, check=True, text=True, capture_output=True)
    print(result.stdout)
